# Semana 3 — LAB estructural

Notebook principal para ejecutar y revisar las partes A–D: carga viva, sismo pseudoestático, superposición y capacidad HA.

Los scripts de cálculo se mantienen como módulos reutilizables; este notebook organiza su ejecución y muestra las verificaciones.

## 0. Preparación

In [ ]:
from pathlib import Path
import json, csv, subprocess, sys
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'results'
print('Carpeta de trabajo:', ROOT)


## 1. Parámetros y sección Fiber

In [ ]:
cfg = json.loads((ROOT / 'parametros.json').read_text(encoding='utf-8'))
col = cfg['columna']
for k in ['b_m','h_m','recubrimiento_hasta_estribo_m','recubrimiento_al_centro_barra_m','diametro_m','estribo_diametro_m','estribo_spacing_m','fc_MPa','fy_MPa']:
    print(f'{k}: {col[k]}')
print('Armadura longitudinal: 16 barras Ø22')


In [ ]:
# Discretización Fiber guardada por el cálculo
rows = list(csv.DictReader((OUT/'fibras.csv').open(encoding='utf-8-sig')))
concrete = [r for r in rows if int(float(r['material'])) == 1]
steel = [r for r in rows if int(float(r['material'])) == 2]
print(f'Fibras de hormigón: {len(concrete)}')
print(f'Fibras de acero: {len(steel)}')
fig, ax = plt.subplots(figsize=(5,5))
ax.scatter([float(r['z_m']) for r in concrete],[float(r['y_m']) for r in concrete],s=2,label='Hormigón')
ax.scatter([float(r['z_m']) for r in steel],[float(r['y_m']) for r in steel],s=35,label='Acero')
ax.set_aspect('equal'); ax.set_xlabel('z [m]'); ax.set_ylabel('y [m]'); ax.legend(); ax.grid(alpha=.2)
plt.show()


## 2. Ejecución de los casos base

In [ ]:
# Ejecuta G, Q, EX, EY, R y las verificaciones de masa/capacidad.
result = subprocess.run([sys.executable, str(ROOT/'ejecutar.py')], cwd=ROOT, text=True, capture_output=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError('La ejecución falló')


## Parte A — carga viva

In [ ]:
transfers = list(csv.DictReader((OUT/'transferencia_Q.csv').open(encoding='utf-8-sig')))
qA = sum(float(r['Q_kN']) for r in transfers)
q_area = sum(float(r['q_Q_kN_m2'])*float(r['area_m2']) for r in transfers)
print(f'Q transferida = {qA:.6f} kN')
print(f'q_Q · A       = {q_area:.6f} kN')
print(f'Error          = {qA-q_area:.3e} kN')


## Parte B — sismo pseudoestático

In [ ]:
floors = list(csv.DictReader((OUT/'sismo_pisos.csv').open(encoding='utf-8-sig')))
for case in ['EX','EY']:
    rows = list(csv.DictReader((OUT/'sismo_por_piso.csv').open(encoding='utf-8-sig')))
    lateral = sum(abs(float(r['Fx_kN'] if case=='EX' else r['Fy_kN'])) for r in rows)
    print(case, 'fuerza lateral aplicada/reacción total ≈', lateral, 'kN')
print('Aceleración sísmica:', cfg['aceleracion_fraccion_g'], 'g')
print('Masa sísmica: G +', cfg['fraccion_Q_masa'], 'Q')


## Parte C — superposición

In [ ]:
summary = json.loads((OUT/'resumen_global.json').read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('La combinación R se contrasta con la corrida explícita en verificar_reparto_combinacion.py.')


## Parte D — capacidad HA

In [ ]:
pm = list(csv.DictReader((OUT/'PM_puntos.csv').open(encoding='utf-8-sig')))
for i, row in enumerate(pm, 1):
    print(f'Punto {i}: P={float(row["P_kN"]):.1f} kN, M={float(row["M_kNm"]):.1f} kN·m')
phi = list(csv.DictReader((OUT/'momento_curvatura.csv').open(encoding='utf-8-sig')))
fig, ax = plt.subplots(figsize=(7,4))
byP = {}
for r in phi: byP.setdefault(r['P_objetivo_kN'], []).append(r)
for p, rowsP in byP.items():
    ax.plot([float(r['phi_1_m']) for r in rowsP],[float(r['M_kNm']) for r in rowsP],label=f'P={float(p):.0f} kN')
ax.set_xlabel('Curvatura φ [1/m]'); ax.set_ylabel('Momento M [kN·m]'); ax.set_title('Momento–curvatura — Fiber Section'); ax.grid(alpha=.2); ax.legend(); plt.show()


## Reproducibilidad

Desde una terminal ubicada en `P1L3`, ejecutar `python ejecutar.py`. Los resultados se escriben en `P1L3/results`. Para abrir este archivo: `jupyter notebook notebooks/Semana3_LAB.ipynb`.